In [13]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

In [14]:
df = pd.read_csv('naoEstacionaria\\vendas_sazonal.csv')

COLUNA_DATA = 'data'
COLUNA_VALOR = 'vendas'

# Limpeza e conversão
# df = df.drop(columns=['Open', 'Low', 'Close', 'Volume'], errors='ignore')
df[COLUNA_DATA] = pd.to_datetime(df[COLUNA_DATA])
df = df.sort_values(COLUNA_DATA)
df.set_index(COLUNA_DATA, inplace=True)

# Agrupamos semanalmente pela média do valor Vendas para reduzir o ruído
df_weekly = df[[COLUNA_VALOR]].resample('W').mean()

nulos = df_weekly.isnull().sum()
print("Valores nulos por coluna na base semanal:")
print(nulos)

if nulos.sum() > 0:
    df_weekly = df_weekly.interpolate(method='linear')
    print("x Valores nulos preenchidos via interpolação linear.")

display(df_weekly.head())
print("\nEstatísticas descritivas da base semanal:")
display(df_weekly.describe())

print("\n--- 4.1 Apresentação da Base ---")
print("Origem: Base de Vendas Sazonais")
print("Contexto: Série temporal para modelagem.")
print(f"Variável Temporal: {COLUNA_DATA}")
print(f"Variável Analisada: {COLUNA_VALOR}")
print("Frequência: Semanal")
print(f"Período: de {df_weekly.index[0].date()} a {df_weekly.index[-1].date()}")
print(f"Quantidade de observações: {len(df_weekly)}")
print("Possíveis problemas: Presença de nulos resolvidos por interpolação linear.")


Valores nulos por coluna na base semanal:
vendas    0
dtype: int64


,vendas
data,
2024-01-07,113.633264
2024-01-14,105.532278
2024-01-21,116.687490
2024-01-28,102.613953
2024-02-04,101.323908



Estatísticas descritivas da base semanal:


,vendas
count,105.000000
mean,100.105423
std,11.693322
min,77.391066
25%,90.043301
50%,100.036673
75%,110.187628
max,122.417010



--- 4.1 Apresentação da Base ---
Origem: Base de Vendas Sazonais
Contexto: Série temporal para modelagem.
Variável Temporal: data
Variável Analisada: vendas
Frequência: Semanal
Período: de 2024-01-07 a 2026-01-04
Quantidade de observações: 105
Possíveis problemas: Presença de nulos resolvidos por interpolação linear.


In [15]:
# Divisão temporal (80/20)
train_size = int(len(df_weekly) * 0.8)
treino = df_weekly[COLUNA_VALOR].iloc[:train_size]
teste = df_weekly[COLUNA_VALOR].iloc[train_size:]

print(f"Total de observações: {len(df_weekly)}")
print(f"Treino: {len(treino)} semanas (de {treino.index[0].date()} a {treino.index[-1].date()})")
print(f"Teste: {len(teste)} semanas (de {teste.index[0].date()} a {teste.index[-1].date()})")


Total de observações: 105
Treino: 84 semanas (de 2024-01-07 a 2025-08-10)
Teste: 21 semanas (de 2025-08-17 a 2026-01-04)


1. Função – Média Histórica

Calcula a média de uma janela de valores anteriores.

In [16]:
# 1. MÉDIA HISTÓRICA 
def media_historica(train, test):
    history = list(train)
    preds = []

    for real in test:
        pred = np.mean(history)

        preds.append(pred)
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds

2. Função – Média Móvel Simples (SMA)

Calcula a média de uma janela de valores anteriores.

A **média móvel (moving average)** em um cálculo que projeta a **melhor janela** para calcular a média dos **valores consecutivos** e ir “andando” com essa janela ao longo.


In [17]:
def sma(train, test, janela):
    history = list(train)
    preds = []

    for real in test:
        pred = np.mean(history[-janela:])

        preds.append(pred)
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds


def testar_janelas_sma(train, test):
    resultados = []

    for janela in range(2, len(train) + 1):
        mae, preds = sma(train, test, janela)

        resultados.append({
            "Janela": janela,
            "MAE": mae,
            "Previsoes": preds
        })

    df_resultados = pd.DataFrame(resultados)
    df_resultados = df_resultados.sort_values("MAE").reset_index(drop=True)

    return df_resultados


def melhor_sma(train, test):
    df_janelas = testar_janelas_sma(train, test)

    melhor_janela = int(df_janelas.iloc[0]["Janela"])
    melhor_mae = df_janelas.iloc[0]["MAE"]
    melhores_preds = df_janelas.iloc[0]["Previsoes"]

    return melhor_mae, melhores_preds, melhor_janela, df_janelas

3. Função – Média Acumulada

Calcula a média de todos os valores até aquele momento.

In [18]:
def media_acumulada(train, test):
    history = list(train)
    preds = []

    for real in test:
        pred = np.mean(history)

        preds.append(pred)
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds

4. Função – Média Móvel Exponencial (EMA)

Dá mais peso aos valores mais recentes.

In [19]:
def ema(train, test, alpha):
    history = list(train)

    ema_val = history[0]

    for valor in history[1:]:
        ema_val = alpha * valor + (1 - alpha) * ema_val

    preds = []

    for real in test:
        pred = ema_val

        preds.append(pred)

        ema_val = alpha * real + (1 - alpha) * ema_val
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds


def testar_alphas_ema(train, test):
    resultados = []

    alphas = np.arange(0.1, 1.0, 0.1)

    for alpha in alphas:
        alpha = round(alpha, 1)

        mae, preds = ema(train, test, alpha)

        resultados.append({
            "Alpha": alpha,
            "MAE": mae,
            "Previsoes": preds
        })

    df_resultados = pd.DataFrame(resultados)
    df_resultados = df_resultados.sort_values("MAE").reset_index(drop=True)

    return df_resultados


def melhor_ema(train, test):
    df_alphas = testar_alphas_ema(train, test)

    melhor_alpha = df_alphas.iloc[0]["Alpha"]
    melhor_mae = df_alphas.iloc[0]["MAE"]
    melhores_preds = df_alphas.iloc[0]["Previsoes"]

    return melhor_mae, melhores_preds, melhor_alpha, df_alphas

5. Taxa de Variação (ROC)

Aqui o erro compara valor real vs taxa prevista anterior.

In [20]:
def taxa_variacao(train, test, janela):
    history = list(train)
    preds = []

    for real in test:
        x_t = history[-1]
        x_tk = history[-janela - 1]

        if x_tk == 0:
            pred = x_t
        else:
            taxa = (x_t - x_tk) / x_tk
            pred = x_t * (1 + taxa)

        preds.append(pred)
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds


def testar_janelas_taxa_variacao(train, test):
    resultados = []

    for janela in range(1, len(train)):
        mae, preds = taxa_variacao(train, test, janela)

        resultados.append({
            "Janela": janela,
            "MAE": mae,
            "Previsoes": preds
        })

    df_resultados = pd.DataFrame(resultados)
    df_resultados = df_resultados.sort_values("MAE").reset_index(drop=True)

    return df_resultados


def melhor_taxa_variacao(train, test):
    df_janelas = testar_janelas_taxa_variacao(train, test)

    melhor_janela = int(df_janelas.iloc[0]["Janela"])
    melhor_mae = df_janelas.iloc[0]["MAE"]
    melhores_preds = df_janelas.iloc[0]["Previsoes"]

    return melhor_mae, melhores_preds, melhor_janela, df_janelas

6. Seasonal Naive

In [21]:
def seasonal_naive(train, test, periodo):
    history = list(train)
    preds = []

    for real in test:
        pred = history[-periodo]

        preds.append(pred)
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds


def testar_periodos_seasonal_naive(train, test):
    resultados = []

    for periodo in range(2, len(train) + 1):
        mae, preds = seasonal_naive(train, test, periodo)

        resultados.append({
            "Periodo": periodo,
            "MAE": mae,
            "Previsoes": preds
        })

    df_resultados = pd.DataFrame(resultados)
    df_resultados = df_resultados.sort_values("MAE").reset_index(drop=True)

    return df_resultados


def melhor_seasonal_naive(train, test):
    df_periodos = testar_periodos_seasonal_naive(train, test)

    melhor_periodo = int(df_periodos.iloc[0]["Periodo"])
    melhor_mae = df_periodos.iloc[0]["MAE"]
    melhores_preds = df_periodos.iloc[0]["Previsoes"]

    return melhor_mae, melhores_preds, melhor_periodo, df_periodos

7. DELTA

In [22]:
def delta(train, test, janela):
    history = list(train)
    preds = []

    for real in test:
        x_t = history[-1]
        x_tk = history[-janela - 1]

        delta = (x_t - x_tk) / janela
        pred = x_t + delta

        preds.append(pred)
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds


def testar_janelas_delta(train, test):
    resultados = []

    for janela in range(1, len(train)):
        mae, preds = delta(train, test, janela)

        resultados.append({
            "Janela": janela,
            "MAE": mae,
            "Previsoes": preds
        })

    df_resultados = pd.DataFrame(resultados)
    df_resultados = df_resultados.sort_values("MAE").reset_index(drop=True)

    return df_resultados


def melhor_delta(train, test):
    df_janelas = testar_janelas_delta(train, test)

    melhor_janela = int(df_janelas.iloc[0]["Janela"])
    melhor_mae = df_janelas.iloc[0]["MAE"]
    melhores_preds = df_janelas.iloc[0]["Previsoes"]

    return melhor_mae, melhores_preds, melhor_janela, df_janelas

In [23]:
# =========================================================
# RODANDO TODOS OS BASE MODELS
# =========================================================

mae_media_historica, pred_media_historica = media_historica(treino, teste)

mae_media_acumulada, pred_media_acumulada = media_acumulada(treino, teste)

mae_sma, pred_sma, melhor_janela_sma, df_sma_janelas = melhor_sma(treino, teste)

mae_ema, pred_ema, melhor_alpha_ema, df_ema_alphas = melhor_ema(treino, teste)

mae_taxa_variacao, pred_taxa_variacao, melhor_janela_taxa, df_taxa_janelas = melhor_taxa_variacao(treino, teste)

mae_seasonal_naive, pred_seasonal_naive, melhor_periodo_seasonal, df_seasonal_periodos = melhor_seasonal_naive(treino, teste)

mae_delta, pred_delta, melhor_janela_delta, df_delta_janelas = melhor_delta(treino, teste)

In [24]:
# =========================================================
# COMPARAÇÃO FINAL
# =========================================================

resultados_base_models = pd.DataFrame({
    "Modelo": [
        "Média Histórica",
        "Média Acumulada",
        "SMA",
        "EMA",
        "Taxa de Variação",
        "Seasonal Naive",
        "Delta"
    ],
    "Melhor Parâmetro": [
        "-",
        "-",
        f"janela={melhor_janela_sma}",
        f"alpha={melhor_alpha_ema}",
        f"janela={melhor_janela_taxa}",
        f"periodo={melhor_periodo_seasonal}",
        f"janela={melhor_janela_delta}"
    ],
    "MAE": [
        mae_media_historica,
        mae_media_acumulada,
        mae_sma,
        mae_ema,
        mae_taxa_variacao,
        mae_seasonal_naive,
        mae_delta
    ]
})

In [25]:
comparacao_previsoes = pd.DataFrame({
    "Real": teste.values,
    "Média Histórica": pred_media_historica,
    "Média Acumulada": pred_media_acumulada,
    "SMA": pred_sma,
    "EMA": pred_ema,
    "Taxa de Variação": pred_taxa_variacao,
    "Seasonal Naive": pred_seasonal_naive,
    "Delta": pred_delta
}, index=teste.index)

comparacao_previsoes

,Real,Média Histórica,Média Acumulada,SMA,EMA,Taxa de Variação,Seasonal Naive,Delta
data,,,,,,,,
2025-08-17,93.158882,100.134347,100.134347,96.898475,85.688351,86.231361,93.071849,83.210656
2025-08-24,90.449059,100.052283,100.052283,98.315174,91.664776,93.245996,90.027159,92.946149
2025-08-31,113.767540,99.940617,99.940617,98.466528,90.692202,90.872935,113.892640,90.857098
2025-09-07,110.187628,100.099547,100.099547,101.320171,109.152473,113.642578,109.587441,114.559606
2025-09-14,122.417010,100.214185,100.214185,101.680904,109.980597,110.791102,121.367392,110.911026
2025-09-21,100.930629,100.463655,100.463655,103.048620,119.929727,123.475705,102.581403,122.681029
2025-09-28,100.113579,100.468843,100.468843,101.459224,104.730448,99.306419,101.988608,100.792526
2025-10-05,79.273936,100.464939,100.464939,100.511884,101.036953,98.273021,82.947453,99.412200
2025-10-12,90.043301,100.234602,100.234602,98.541257,83.626540,75.763109,84.948545,78.455813
